# Session 2: count the steps of five sorting algorithms
**A. Real-world problem.** A shop has 1,000,000 orders to put in date order. Two programs give
the same sorted answer; one finishes in a blink and the other keeps the machine busy for minutes.
The difference is the *number of steps*, so we count steps instead of guessing.

**Plan.** (1) write bubble, selection, insertion, merge and quick sort with counters,
(2) test that each one is correct, (3) measure best, average and worst inputs,
(4) compare the counts with the formulas n(n-1)/2 and n log2 n, (5) check stability.

In [ ]:
import math
import pathlib
import random

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the five algorithms
Every function returns `(sorted_list, comparisons, moves)` and never changes its input.
A *comparison* looks at two items. A *move* is one write of an item: a swap in bubble, selection
and quick sort, a one-place shift in insertion sort, an item copied into the merged list in merge sort.
`key` says what to sort by (default: the item itself); it is used for the stability test.

In [ ]:
def _key(key):
    return key if key is not None else (lambda x: x)


def bubble_sort(values, key=None):
    k = _key(key)
    a = list(values)
    comps = moves = 0
    for end in range(len(a) - 1, 0, -1):
        swapped = False
        for j in range(end):
            comps += 1
            if k(a[j]) > k(a[j + 1]):
                a[j], a[j + 1] = a[j + 1], a[j]
                moves += 1
                swapped = True
        if not swapped:
            break
    return a, comps, moves


def selection_sort(values, key=None):
    k = _key(key)
    a = list(values)
    comps = moves = 0
    for i in range(len(a) - 1):
        m = i
        for j in range(i + 1, len(a)):
            comps += 1
            if k(a[j]) < k(a[m]):
                m = j
        if m != i:
            a[i], a[m] = a[m], a[i]
            moves += 1
    return a, comps, moves


def insertion_sort(values, key=None):
    k = _key(key)
    a = list(values)
    comps = moves = 0
    for i in range(1, len(a)):
        card = a[i]
        j = i - 1
        while j >= 0:
            comps += 1
            if k(a[j]) > k(card):
                a[j + 1] = a[j]
                moves += 1
                j -= 1
            else:
                break
        a[j + 1] = card
    return a, comps, moves


def merge_sort(values, key=None):
    k = _key(key)
    counts = [0, 0]

    def sort(xs):
        if len(xs) <= 1:
            return xs
        mid = len(xs) // 2
        left, right = sort(xs[:mid]), sort(xs[mid:])
        out, i, j = [], 0, 0
        while i < len(left) and j < len(right):
            counts[0] += 1
            if k(right[j]) < k(left[i]):      # strict: on a tie the left item goes first (stable)
                out.append(right[j])
                j += 1
            else:
                out.append(left[i])
                i += 1
            counts[1] += 1
        for rest in (left[i:], right[j:]):
            for x in rest:
                out.append(x)
                counts[1] += 1
        return out

    return sort(list(values)), counts[0], counts[1]


def quick_sort(values, key=None):
    """First item as pivot, Lomuto partition, explicit stack (no deep recursion)."""
    k = _key(key)
    a = list(values)
    comps = moves = 0
    stack = [(0, len(a) - 1)]
    while stack:
        lo, hi = stack.pop()
        if lo >= hi:
            continue
        pivot = k(a[lo])
        i = lo
        for j in range(lo + 1, hi + 1):
            comps += 1
            if k(a[j]) < pivot:
                i += 1
                a[i], a[j] = a[j], a[i]
                moves += 1
        a[lo], a[i] = a[i], a[lo]
        moves += 1
        stack.append((lo, i - 1))
        stack.append((i + 1, hi))
    return a, comps, moves


ALGORITHMS = {"bubble": bubble_sort, "selection": selection_sort, "insertion": insertion_sort,
              "merge": merge_sort, "quick": quick_sort}

## Step 2: correctness comes first
A fast wrong sort is useless. A result is correct when (1) it is in order and (2) it contains
exactly the same items as the input (same multiset). We compare against Python's `sorted`.

In [ ]:
def is_sorted(values):
    return all(values[i] <= values[i + 1] for i in range(len(values) - 1))


def is_correct(result, original):
    return is_sorted(result) and sorted(result) == sorted(original)


def make_input(kind, n, seed=0):
    """kind: 'sorted', 'reversed' or 'random' (a shuffle of 0..n-1)."""
    if kind == "sorted":
        return list(range(n))
    if kind == "reversed":
        return list(range(n - 1, -1, -1))
    rng = random.Random(seed)
    xs = list(range(n))
    rng.shuffle(xs)
    return xs

## Step 3: best, average and worst inputs
The step count depends on the input. 'Average' here is the mean over 20 random shuffles.

In [ ]:
def measure(algo, kind, n, trials=20):
    """(comparisons, moves) for one input kind; random is averaged over `trials` shuffles."""
    fn = ALGORITHMS[algo]
    if kind != "random":
        _, c, m = fn(make_input(kind, n))
        return float(c), float(m)
    cs, ms = [], []
    for seed in range(trials):
        _, c, m = fn(make_input("random", n, seed))
        cs.append(c)
        ms.append(m)
    return sum(cs) / trials, sum(ms) / trials

## Step 4: the formulas
* bubble, selection, insertion (worst case): about n(n-1)/2 = n^2/2 comparisons.
* merge sort: between (n/2) log2 n and n log2 n - n + 1 comparisons.
* Insertion sort on a reversed list does exactly n(n-1)/2 comparisons and n(n-1)/2 shifts.

In [ ]:
def quad(n):
    return n * (n - 1) / 2


def nlogn(n):
    return n * math.log2(n)


def cost_ratio(n):
    """How many times more comparisons n(n-1)/2 needs than n log2 n."""
    return quad(n) / nlogn(n)


def seconds(steps, steps_per_second=1e9):
    return steps / steps_per_second

## Step 5: nearly sorted data and stability
A *stable* sort keeps equal items in their original order. We test it with (key, position) records.

In [ ]:
def nearly_sorted(n, swaps, seed=1):
    """0..n-1 in order, then `swaps` random exchanges of items at most 3 places apart."""
    rng = random.Random(seed)
    a = list(range(n))
    for _ in range(swaps):
        i = rng.randrange(n - 3)
        j = i + rng.randint(1, 3)
        a[i], a[j] = a[j], a[i]
    return a


def inversions(values):
    """Number of pairs in the wrong order (merge-sort style count, n log n steps)."""
    def rec(xs):
        if len(xs) <= 1:
            return xs, 0
        mid = len(xs) // 2
        left, a = rec(xs[:mid])
        right, b = rec(xs[mid:])
        out, i, j, c = [], 0, 0, a + b
        while i < len(left) and j < len(right):
            if right[j] < left[i]:
                out.append(right[j])
                j += 1
                c += len(left) - i
            else:
                out.append(left[i])
                i += 1
        return out + left[i:] + right[j:], c
    return rec(list(values))[1]


def is_stable(algo, trials=200, n=12, seed=5):
    """True if the algorithm keeps equal-key records in input order on every random test."""
    rng = random.Random(seed)
    fn = ALGORITHMS[algo]
    for _ in range(trials):
        recs = [(rng.randrange(4), i) for i in range(n)]      # (key, original position)
        out = fn(recs, key=lambda r: r[0])[0]
        if out != sorted(recs, key=lambda r: r[0]):           # Python's sorted is stable
            return False
    return True

## Step 6: see it
Left: comparisons of the five methods on sorted, random and reversed lists of 128 items (log scale).
Right: insertion sort on reversed lists matches n(n-1)/2 exactly.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    n = 128
    kinds = ["sorted", "random", "reversed"]
    print("n = %d, comparisons / moves" % n)
    fig, ax = plt.subplots(figsize=(7, 4))
    width = 0.25
    for i, kind in enumerate(kinds):
        vals = []
        for name in ALGORITHMS:
            c, m = measure(name, kind, n)
            vals.append(max(c, 1))
            print("  %-9s %-9s comps %8.1f moves %8.1f" % (name, kind, c, m))
        ax.bar([x + (i - 1) * width for x in range(5)], vals, width, label=kind)
    ax.set_xticks(range(5))
    ax.set_xticklabels(list(ALGORITHMS))
    ax.set_yscale("log")
    ax.set_ylabel("comparisons (log scale)")
    ax.set_xlabel("algorithm (128 items)")
    ax.set_title("Comparisons depend on the algorithm and on the input")
    ax.legend(title="input")
    ax.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    fig.savefig(OUT / "comparisons_by_input.png", dpi=120)

    ns = list(range(2, 61, 2))
    meas = [insertion_sort(make_input("reversed", m))[1] for m in ns]
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(ns, [quad(m) for m in ns], label="n(n-1)/2")
    ax2.plot(ns, meas, "o", ms=4, label="insertion sort, reversed input")
    ax2.set_xlabel("n (items)")
    ax2.set_ylabel("comparisons")
    ax2.set_title("Measured counts sit on the formula")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "insertion_vs_formula.png", dpi=120)

    big = 1_000_000
    print("n = 1,000,000: n(n-1)/2 = %.4g, n log2 n = %.4g, ratio %.0f" % (quad(big), nlogn(big), cost_ratio(big)))
    print("at 1e9 steps per second: %.1f s versus %.3f s" % (seconds(quad(big)), seconds(nlogn(big))))
    for name in ALGORITHMS:
        print("stable?", name, is_stable(name))
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Run `measure` for each algorithm at n = 64 and n = 128 and write down by what factor the comparisons of bubble sort and of merge sort grow when n doubles.
2. **Stretch.** Write a `shaker_sort` (bubble sort that alternates direction) with the same counters, test it with `is_correct`, and compare it with bubble sort on a list where the smallest item is last.
3. **Explain.** Explain why a stable sort matters when you sort a table of students first by name and then by grade.

## Assessment
- **A. Conceptual.** Two programs sort the same 1,000,000 records and give identical answers. Why can the more efficient algorithm be hugely faster on this large dataset, even though both are correct?
- **B. Numerical.** Predict how many comparisons insertion sort needs on a reversed list of 100 items, and how many item shifts. Then say what the answer would be for a sorted list of 100 items.
- **C. Graph and simulation interpretation.** In the bar figure for 128 items, selection sort has the same height for sorted, random and reversed input, while insertion sort does not. Explain this, and give the numbers for n = 1000 sorted and reversed.
- **D. Design and optimisation challenge.** Design: a log file of 1000 lines is nearly sorted by time, with 10 neighbouring lines swapped at random. Choose between insertion sort, merge sort and quick sort (first item as pivot) and justify your choice with step counts.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    # correctness of every algorithm on several inputs, including empty, one item and duplicates
    cases = [[], [1], [2, 1], [3, 3, 3], [5, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]]
    cases += [L.make_input(k, 50, 3) for k in ("sorted", "reversed", "random")]
    good = True
    for name, fn in L.ALGORITHMS.items():
        for data in cases:
            snapshot = list(data)
            out = fn(data)[0]
            good &= L.is_correct(out, data) and out == sorted(data) and data == snapshot
    checks.append(("all five sorts return a sorted list with the same items and leave the input unchanged", good))

    # worked example (Section E): n = 1,000,000
    n = 1_000_000
    checks.append(("worked example: n = 1,000,000 needs 499,999,500,000 comparisons for n(n-1)/2",
                   L.quad(n) == 499_999_500_000))
    checks.append(("worked example: n log2 n is about 19,931,569 for n = 1,000,000",
                   abs(L.nlogn(n) - 19_931_569) < 1))
    checks.append(("worked example: the ratio is about 25,086 times",
                   abs(L.cost_ratio(n) - 25_086) < 1))
    checks.append(("worked example: at 1e9 steps per second that is 500 s against 0.02 s",
                   abs(L.seconds(L.quad(n)) - 500.0) < 0.01 and abs(L.seconds(L.nlogn(n)) - 0.0199) < 0.0005))

    # hand calculation: insertion sort on a reversed list of 100
    _, c, m = L.insertion_sort(L.make_input("reversed", 100))
    checks.append(("insertion sort on a reversed list of 100: 4950 = 100*99/2 comparisons and shifts", c == 4950 == m == 100 * 99 // 2))
    checks.append(("insertion sort on [2, 1, 3] takes 2 comparisons and 1 shift", L.insertion_sort([2, 1, 3])[1:] == (2, 1)))
    ok = all(L.insertion_sort(L.make_input("reversed", k))[1] == k * (k - 1) // 2 for k in range(2, 40))
    checks.append(("the formula n(n-1)/2 holds exactly for n = 2 to 39", ok))

    checks.append(("snippet: n = 1000 gives n(n-1)/2 = 499,500 and n log2 n = 9,966 (rounded)",
                   L.quad(1000) == 499_500 and round(L.nlogn(1000)) == 9966))

    # best case, worst case at n = 1000
    n = 1000
    rev = {a: L.ALGORITHMS[a](L.make_input("reversed", n))[1:] for a in L.ALGORITHMS}
    srt = {a: L.ALGORITHMS[a](L.make_input("sorted", n))[1:] for a in L.ALGORITHMS}
    checks.append(("n = 1000 reversed: bubble, selection, insertion and quick all use 499,500 comparisons",
                   all(rev[a][0] == 499_500 for a in ("bubble", "selection", "insertion", "quick"))))
    checks.append(("n = 1000 sorted: bubble and insertion need only 999 comparisons and no moves",
                   srt["bubble"] == (999, 0) and srt["insertion"] == (999, 0)))
    checks.append(("selection sort does n(n-1)/2 comparisons even on sorted input", srt["selection"][0] == 499_500))
    checks.append(("selection sort moves at most n - 1 = 999 items", all(rev[a][1] <= 999 for a in ("selection",))))
    checks.append(("quick sort with the first item as pivot is quadratic on sorted input (499,500 comparisons)", srt["quick"][0] == 499_500))

    # merge sort bounds and exact values at n = 1024
    n = 1024
    lo, hi = n // 2 * math.log2(n), n * math.log2(n) - n + 1
    ms = L.merge_sort(L.make_input("sorted", n))
    mr = L.merge_sort(L.make_input("reversed", n))
    mx = L.merge_sort(L.make_input("random", n, 0))
    checks.append(("merge sort n = 1024: best 5120 = (n/2) log2 n, worst-case bound 9217 = n log2 n - n + 1",
                   ms[1] == 5120 == lo and hi == 9217))
    checks.append(("merge sort comparisons on sorted, reversed and random lists all lie between 5120 and 9217",
                   all(lo <= r[1] <= hi for r in (ms, mr, mx))))
    checks.append(("merge sort makes exactly n log2 n = 10240 item moves for n = 1024", ms[2] == mr[2] == mx[2] == 10_240))
    qa = L.measure("quick", "random", 1024)[0]
    checks.append(("quick sort random average at n = 1024 is below the 2 n ln n bound (14,196)", abs(2 * n * math.log(n) - 14196) < 1 and n * math.log2(n) * 0.9 < qa < 2 * n * math.log(n)))

    # n squared versus n log n grows apart
    r = [L.measure("bubble", "random", k)[0] / L.measure("merge", "random", k)[0] for k in (64, 128, 256)]
    checks.append(("bubble divided by merge comparisons grows with n", r[0] < r[1] < r[2]))

    # design problem: nearly sorted data
    data = L.nearly_sorted(1000, 10)
    inv = L.inversions(data)
    ins = L.insertion_sort(data)
    mg = L.merge_sort(data)
    checks.append(("design: nearly sorted list of 1000 has only 28 inversions", inv == 28))
    checks.append(("design: insertion sort needs 1027 = (n - 1) + inversions comparisons", ins[1] == 999 + inv == 1027))
    checks.append(("design: merge sort needs 4947 comparisons, more than 4 times insertion sort", mg[1] == 4947 and mg[1] > 4 * ins[1]))
    checks.append(("design: quick sort (first pivot) is the wrong choice here: over 400,000 comparisons",
                   L.quick_sort(data)[1] > 400_000))
    checks.append(("inversions: sorted 0, reversed n(n-1)/2, [2,1,3] has 1",
                   L.inversions(list(range(10))) == 0 and L.inversions(list(range(10, 0, -1))) == 45 and L.inversions([2, 1, 3]) == 1))

    # stability
    stable = {a: L.is_stable(a) for a in L.ALGORITHMS}
    checks.append(("stability: bubble, insertion and merge are stable", stable["bubble"] and stable["insertion"] and stable["merge"]))
    checks.append(("stability: selection and quick sort are not stable", not stable["selection"] and not stable["quick"]))
    recs = [(2, "a"), (2, "b"), (1, "c")]
    checks.append(("selection sort turns [(2,a),(2,b),(1,c)] into (1,c),(2,b),(2,a): equal keys swapped",
                   L.selection_sort(recs, key=lambda r: r[0])[0] == [(1, "c"), (2, "b"), (2, "a")]))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
